In [2]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

processed_dir = project_root / "data" / "processed"
results_root = project_root / "results"

paysim_train = pd.read_csv(processed_dir / "paysim_train.csv")
paysim_val = pd.read_csv(processed_dir / "paysim_val.csv")
paysim_test = pd.read_csv(processed_dir / "paysim_test.csv")

In [3]:
import importlib
import test01

importlib.reload(test01)

print("Loaded from:", test01.__file__)

Loaded from: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src\test01.py


In [4]:
functions_to_check = [
    "engineer_features",
    "get_feature_config",
    "prepare_dataframes",
    "build_preprocessor",
    "compute_scale_pos_weight",
    "build_xgb_pipeline",
    "fit_model",
    "investigate_flagged_fraud",
    "evaluate_model",
    "search_thresholds",
    "compute_cost_table",
    "evaluate_with_threshold",
    "compute_expected_cost_from_cm",
    "save_experiment_outputs",
]

for func in functions_to_check:
    print(func, "✅" if hasattr(test01, func) else "❌ missing")

engineer_features ✅
get_feature_config ✅
prepare_dataframes ✅
build_preprocessor ✅
compute_scale_pos_weight ✅
build_xgb_pipeline ✅
fit_model ✅
investigate_flagged_fraud ✅
evaluate_model ✅
search_thresholds ✅
compute_cost_table ✅
evaluate_with_threshold ✅
compute_expected_cost_from_cm ✅
save_experiment_outputs ✅


In [5]:
sample_raw = paysim_train.head(10).copy()

sample_eng = test01.engineer_features(sample_raw)

new_cols = [
    "balance_delta_orig",
    "balance_delta_dest",
    "amount_to_balance_ratio",
    "orig_balance_zeroed",
]

print(sample_eng[new_cols].head())
print("\nNew columns exist:")
print(sample_eng[new_cols].dtypes)

   balance_delta_orig  balance_delta_dest  amount_to_balance_ratio  \
0           -30075.63            30075.63                 0.406356   
1                0.00                0.00              1235.680000   
2                0.00                0.00              3382.270000   
3            -4181.40                0.00                 0.124310   
4           114104.26          -114104.26                 0.041106   

   orig_balance_zeroed  
0                    0  
1                    1  
2                    1  
3                    0  
4                    0  

New columns exist:
balance_delta_orig         float64
balance_delta_dest         float64
amount_to_balance_ratio    float64
orig_balance_zeroed          int64
dtype: object


In [6]:
target_col, categorical_features, numeric_features, feature_cols = test01.get_feature_config()

print("Target:", target_col)
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("All feature columns:", feature_cols)

Target: isFraud
Categorical features: ['type']
Numeric features: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'balance_delta_orig', 'balance_delta_dest', 'amount_to_balance_ratio', 'orig_balance_zeroed']
All feature columns: ['type', 'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'balance_delta_orig', 'balance_delta_dest', 'amount_to_balance_ratio', 'orig_balance_zeroed']


In [7]:
data = test01.prepare_dataframes(
    paysim_train,
    paysim_val,
    paysim_test
)

print(data.keys())

print("X_train shape:", data["X_train"].shape)
print("y_train shape:", data["y_train"].shape)
print("X_val shape:", data["X_val"].shape)
print("y_val shape:", data["y_val"].shape)
print("X_test shape:", data["X_test"].shape)
print("y_test shape:", data["y_test"].shape)

print("\nFeature columns:")
print(data["feature_cols"])

print("\nFirst rows of X_train:")
display(data["X_train"].head())

dict_keys(['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test', 'categorical_features', 'numeric_features', 'feature_cols', 'target_col'])
X_train shape: (4453834, 12)
y_train shape: (4453834,)
X_val shape: (636262, 12)
y_val shape: (636262,)
X_test shape: (1272524, 12)
y_test shape: (1272524,)

Feature columns:
['type', 'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'balance_delta_orig', 'balance_delta_dest', 'amount_to_balance_ratio', 'orig_balance_zeroed']

First rows of X_train:


,type,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,balance_delta_orig,balance_delta_dest,amount_to_balance_ratio,orig_balance_zeroed
0,1,226,30075.63,74012.0,43936.37,0.0,30075.63,0,-30075.63,30075.63,0.406356,0
1,3,256,1235.68,0.0,0.00,0.0,0.00,0,0.00,0.00,1235.680000,1
2,3,257,3382.27,0.0,0.00,0.0,0.00,0,0.00,0.00,3382.270000,1
3,3,209,4181.40,33636.0,29454.60,0.0,0.00,0,-4181.40,0.00,0.124310,0
4,0,161,114104.26,2775869.4,2889973.66,918761.6,804657.34,0,114104.26,-114104.26,0.041106,0


In [8]:
scale_pos_weight = test01.compute_scale_pos_weight(data["y_train"])

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 773.7145590537485


In [9]:
preprocessor = test01.build_preprocessor(
    numeric_features=data["numeric_features"],
    categorical_features=data["categorical_features"],
)

X_small = data["X_train"].head(1000)

X_small_processed = preprocessor.fit_transform(X_small)

print("Original shape:", X_small.shape)
print("Processed shape:", X_small_processed.shape)

Original shape: (1000, 12)
Processed shape: (1000, 16)


In [10]:
preprocessor = test01.build_preprocessor(
    numeric_features=data["numeric_features"],
    categorical_features=data["categorical_features"],
)

model_none = test01.build_xgb_pipeline(
    preprocessor=preprocessor,
    imbalance_method="none"
)

model_weighted = test01.build_xgb_pipeline(
    preprocessor=preprocessor,
    imbalance_method="scale_pos_weight",
    scale_pos_weight=scale_pos_weight
)

model_smote = test01.build_xgb_pipeline(
    preprocessor=preprocessor,
    imbalance_method="smote",
    smote_sampling_strategy=0.1
)

print(model_none)
print(model_weighted)
print(model_smote)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['step', 'amount',
                                                   'oldbalanceOrg',
                                                   'newbalanceOrig',
                                                   'oldbalanceDest',
                                                   'newbalanceDest',
                                                   'isFlaggedFraud',
                                                   'balance_delta_orig',
                                                   'balance_delta_dest',
                                                   'amount_to_balance_ratio',
                                                   'orig_balance_zeroed']),
               